# Self-Supervised Fisheye Rectification — Kaggle training notebook

This notebook trains the `ParametersEstimationModule` from
[MehdiHatab/fisheyeRecti](https://github.com/MehdiHatab/fisheyeRecti)
(the `SelfSupervisedFisheyeRectification` project) on the **NYU Depth V2** dataset from
[Kaggle](https://www.kaggle.com/datasets/soumikrakshit/nyu-depth-v2), which is mounted at

```
/kaggle/input/datasets/soumikrakshit/nyu-depth-v2/nyu_data/data/nyu2_train
```

### What it does
1. Finds the NYU images recursively (images live in scene sub-folders).
2. Uses the model's **single-parameter division model** to generate synthetic fisheye
   images on the fly (the original repo's sparse-matrix warp is replaced with a fast
   `cv2.remap` version).
3. Trains the network to estimate the division-model coefficient `λ` (this is the classic
   self-supervised/reconstruction setup: we know the artificial distortion while training,
   so regression on `λ` is well posed and avoids the collapse of the repo's original
   key-coordinate loss).
4. Saves a checkpoint after every epoch, so you can **stop and resume from the last epoch**.
5. Plots train/val loss.
6. Converts the estimated division-model distortion into an approximate
   **Kannala–Brandt (KB)** parameter set `(fx, fy, cx, cy, k1, k2, k3, k4)`.
7. Runs **inference on your own images** (not training images) and rectifies them.

### Notes / caveats
- The original network only outputs **one scalar** (the division-model `λ`); it is not a
  full KB parameter regressor. The notebook therefore fits a KB polynomial that best matches
  the division-model radial mapping for the predicted `λ` (this is the "if possible" part).
- For a real fisheye photo the predicted `λ` is the model's estimate of the division-model
  coefficient; use the simple division-model rectification for the practical result and the
  KB parameters with OpenCV's `fisheye` module if you need a calibrated camera model.


In [ ]:
UPSTREAM_REPO = "https://github.com/MasakiHosono/SelfSupervisedFisheyeRectification"
LOCAL_DIR = "SelfSupervisedFisheyeRectification-main"
SRC_DIR = None

if os.path.isdir(LOCAL_DIR):
    candidate = os.path.join(LOCAL_DIR, "src")
    if os.path.isdir(candidate):
        SRC_DIR = candidate
        print("Using already-cloned/upstream repo src at:", SRC_DIR)

if SRC_DIR is None:
    print("Local repo src not usable. Re-cloning the upstream SelfSupervisedFisheyeRectification...")
    if os.path.exists(LOCAL_DIR):
        import shutil
        shutil.rmtree(LOCAL_DIR)
    if os.path.exists("repo.zip"):
        os.remove("repo.zip")
    subprocess.run(["wget", "-q", UPSTREAM_REPO + "/archive/refs/heads/main.zip", "-O", "repo.zip"], check=False)
    # Extract, and if the archive wraps it in a subfolder, flatten it.
    subprocess.run(["unzip", "-o", "repo.zip"], check=False)
    # repo.zip from GitHub usually extracts <repo>-main/<files>.
    for d in os.listdir("."):
        if d.endswith("-main") and os.path.isdir(d):
            srcdir = os.path.join(d, "src")
            if os.path.isdir(srcdir):
                if os.path.exists(LOCAL_DIR):
                    shutil.rmtree(LOCAL_DIR)
                os.rename(d, LOCAL_DIR)
                SRC_DIR = srcdir
                print("Extracted and flattened to:", LOCAL_DIR, "-> src:", SRC_DIR)
                break
    if SRC_DIR is None:
        print("WARNING: could not find repo src after download. Training helpers will be provided locally.")

if SRC_DIR:
    sys.path.insert(0, SRC_DIR)
    print("Using model source:", SRC_DIR)
else:
    print("Using notebook-local training helpers (repo src unavailable).")


## 1. Locate the model source
If the zipped project is not already extracted next to this notebook, download it from GitHub.


In [ ]:
if not os.path.isdir(MODEL_DIR):
    print("Model source not found at", MODEL_DIR)
    print("Downloading", REPO_URL)
    subprocess.run(["wget", "-q", REPO_URL, "-O", "repo.zip"], check=False)
    subprocess.run(["unzip", "-o", "repo.zip"], check=False)

if os.path.isdir(MODEL_DIR):
    SRC_DIR = os.path.join(MODEL_DIR, "src")
    sys.path.insert(0, SRC_DIR)
    print("Using model source:", SRC_DIR)
else:
    SRC_DIR = "."
    print("Using current directory as model source (no extracted repo found).")


## 2. Discover the NYU Depth V2 images
Images are in scene sub-folders. The notebook walks the tree recursively and applies a small
heuristic to skip obvious depth/label files. If your Kaggle dataset is mounted at a different
path, set `NYU2_TRAIN_DIR` before running this cell.


In [ ]:
# Discover the NYU Depth V2 images (the helper is defined in the first setup cell).
train_images, val_images, DATA_ROOT = load_nyu_images(
    max_train=CFG.get("MAX_TRAIN_IMAGES"),
    max_val=CFG.get("MAX_VAL_IMAGES"),
    val_ratio=CFG.get("VAL_RATIO", 0.1),
)


## 3. Model
`ParametersEstimationModule` from the repo (encoder → decoder → VGG11 classifier),
with `torch.hub.load` replaced by plain `torchvision.models.vgg11` so it works offline on Kaggle.


In [ ]:
class ParametersEstimationModule(nn.Module):
    def __init__(self, in_channels=3):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 64, 3, 1),
            nn.Conv2d(64, 64, 3, 1),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, 1),
            nn.Conv2d(128, 128, 3, 1),
            nn.MaxPool2d(2, 2),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 2, 2),
            nn.ConvTranspose2d(64, 3, 2, 2),
        )
        try:
            self.vgg = torchvision.models.vgg11(pretrained=False)
        except TypeError:
            self.vgg = torchvision.models.vgg11(weights=None)
        self.vgg.classifier[6] = nn.Linear(self.vgg.classifier[6].in_features, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        x = self.vgg(x)
        return torch.flatten(x)

    def get_transforms(self):
        return transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
        ])

model = ParametersEstimationModule().to(DEVICE)
print(model)


## 4. Fast fisheye effector, dataset and loss
The original repo warps images with a giant sparse matrix and is very slow. This cell keeps the
same single-parameter division model (and the same crop/resize behavior) but uses fast
`cv2.remap` maps so training on Kaggle is practical.


## 4b. Training helpers (distortion schedule, division-model warp, dataset)

The upstream repo ships `getDistortions` plus a sparse-matrix `FisheyeEffector`/`DistortDataset`.
This notebook is meant to run with a fast `cv2.remap` division-model pipeline, so the cell below
defines notebook-local versions of the helpers and a `DistortDataset` that accepts the *image path
lists* produced by the discovery cell (rather than the repo's `<dataset>/train.lst` convention).

If `SRC_DIR` from cell 1 is on `sys.path`, the upstream implementations can be imported instead —
this cell only provides fallbacks, so it never shadows a working upstream import.


In [ ]:
# Notebook-local division-model helpers (fallbacks if upstream repo src is unavailable).
# If the upstream src was imported in cell 1, these names can still be shadowed by the repo's
# versions where useful (e.g. getDistortions), but this cell guarantees a working pipeline.

try:
    from core.functions import getDistortions as _upstream_getDistortions
    def get_distortions(n, random_values=False):
        return _upstream_getDistortions(n, random_values=random_values)
    print('Using upstream getDistortions from core.functions.')
except Exception:
    def get_distortions(n, random_values=False):
        """Return n curriculum distortion values in [-1,1]."""
        interval = 0.9 / max(1, n - 1)
        items = [interval * i for i in range(n)]
        if random_values:
            items = [v - random.random() * interval for v in items]
        return items
    print('Using notebook-local get_distortions.')


def _clipd(d):
    d = float(d)
    if d > 1:
        return 1.0
    if d < -1:
        return -1.0
    return d


def division_forward_vec(xs, ys, r, distortion):
    """Map normalized *original* coords (xs, ys) to *distorted* normalized coords.
    Division model (as used by the repo's FisheyeEffector):
        rho_d = rho / (1 - d * rho^2)
    where (xs, ys) are the original normalized coords and r = sqrt(xs^2 + ys^2).
    """
    d = _clipd(distortion)
    denom = 1.0 - d * r * r
    denom = np.maximum(denom, 1e-8)
    scale = 1.0 / denom
    xd = xs * scale
    yd = ys * scale
    return xd.astype(np.float32), yd.astype(np.float32)


def division_inverse_vec(xs, ys, r, distortion):
    """Inverse of division_forward_vec (distorted -> original normalized coords).
    Useful for rectification builds.
    """
    d = _clipd(distortion)
    denom = 1.0 + d * r * r
    denom = np.maximum(denom, 1e-8)
    scale = 1.0 / denom
    xu = xs * scale
    yu = ys * scale
    return xu.astype(np.float32), yu.astype(np.float32)


class LocalFisheyeEffector:
    """Fast division-model effector backed by cv2.remap (matches the notebook's warp).
    Compatible enough with the repo's FisheyeEffector API for training/DataLoader use.
    """
    def __init__(self, height=720, width=1280, distortion=0.5):
        self.height = int(height)
        self.width = int(width)
        self.distortion = float(distortion)
        self._build_maps()

    def _build_maps(self):
        h, w = self.height, self.width
        f_xy = min(w, h) / 2.0
        cx, cy = w / 2.0, h / 2.0
        xs = (np.arange(w, dtype=np.float32)[None, :] - cx) / f_xy
        ys = (np.arange(h, dtype=np.float32)[:, None] - cy) / f_xy
        r = np.sqrt(xs * xs + ys * ys)
        fwd_xs, fwd_ys = division_forward_vec(xs, ys, r, self.distortion)
        self.map_x = (fwd_xs * f_xy + cx).astype(np.float32)
        self.map_y = (fwd_ys * f_xy + cy).astype(np.float32)

    def getDistortion(self):
        return self.distortion

    def getKeyCoordinates(self):
        return [(0.0, 0.0)]

    def __call__(self, image):
        if isinstance(image, Image.Image):
            image = np.array(image)
        h, w = image.shape[:2]
        if (h, w) != (self.height, self.width):
            image = cv2.resize(image, (self.width, self.height), interpolation=cv2.INTER_LINEAR)
        if image.ndim == 2:
            image = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
        warped = cv2.remap(image, self.map_x, self.map_y, cv2.INTER_LINEAR,
                            borderMode=cv2.BORDER_CONSTANT, borderValue=0)
        return Image.fromarray(warped)


class DistortDataset(torch.utils.data.Dataset):
    """Dataset that uses notebook-local effectors (fast cv2.remap).
    Accepts a list of image paths, which is what the discovery cell produces.
    """
    def __init__(self, image_paths, height=720, width=1280, transform=None,
                 distortions=None, return_distortion=False):
        self.image_paths = list(image_paths)
        self.height = int(height)
        self.width = int(width)
        self.transform = transform
        self.return_distortion = bool(return_distortion)
        self._effectors = []
        self.set_distortions(distortions)

    def set_distortions(self, distortions):
        dist = list(distortions) if distortions is not None else [0.5]
        self._effectors = [LocalFisheyeEffector(height=self.height, width=self.width,
                                                  distortion=d) for d in dist]

    def update_effector(self, distortions=None):
        self.set_distortions(distortions)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        image = Image.open(path).convert('RGB')
        image = image.resize((self.width, self.height), Image.LANCZOS)
        effector = random.choice(self._effectors)
        distorted = effector(image)
        if self.transform is not None:
            distorted = self.transform(distorted)
        if self.return_distortion:
            return distorted, torch.tensor(effector.getDistortion(), dtype=torch.float32)
        return distorted
    



In [ ]:
# For a quick visual sanity check of the distortion augmentation.
# Auto-load NYU images if the discovery/training cells have not run yet; otherwise fall
# back to a synthetic image so this cell never crashes with NameError.
transform = model.get_transforms()
if "train_images" not in globals() or not train_images:
    print("train_images not defined - auto-loading NYU images now.")
    train_images, val_images, DATA_ROOT = load_nyu_images(
        max_train=CFG.get("MAX_TRAIN_IMAGES"),
        max_val=CFG.get("MAX_VAL_IMAGES"),
        val_ratio=CFG.get("VAL_RATIO", 0.1),
    )

if train_images:
    _sample_img = train_images[0]
    _sample = Image.open(_sample_img).convert("RGB").resize((CFG["IMAGE_WIDTH"], CFG["IMAGE_HEIGHT"]))
else:
    print("WARNING: no NYU images found; using a synthetic placeholder for this preview.")
    _rng = np.random.RandomState(SEED)
    _sample = Image.fromarray(_rng.randint(0, 255, (CFG["IMAGE_HEIGHT"], CFG["IMAGE_WIDTH"], 3)).astype(np.uint8))

def _division_warp_preview(img_pil, distortion, in_w, in_h, f_xy=None):
    if f_xy is None:
        f_xy = min(in_w, in_h) / 2.0
    cx, cy = in_w / 2.0, in_h / 2.0
    xs = (np.arange(in_w, dtype=np.float32)[None, :] - cx) / f_xy
    ys = (np.arange(in_h, dtype=np.float32)[:, None] - cy) / f_xy
    r = np.sqrt(xs * xs + ys * ys)
    # division model: distorted = r / (1 - d * r^2)
    denom = 1.0 - distortion * r * r
    denom = np.maximum(denom, 1e-6)
    r_d = r / denom
    scale = r_d / np.maximum(r, 1e-6)
    scale = np.minimum(scale, 1e3)
    xd = xs * scale
    yd = ys * scale
    map_x = (xd * f_xy + cx).astype(np.float32)
    map_y = (yd * f_xy + cy).astype(np.float32)
    src = np.array(img_pil.convert("RGB"))
    warped = cv2.remap(src, map_x, map_y, cv2.INTER_LINEAR,
                       borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    return Image.fromarray(warped)

_fish = _division_warp_preview(_sample, 0.5, CFG["IMAGE_WIDTH"], CFG["IMAGE_HEIGHT"])
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
axs[0].imshow(_sample); axs[0].set_title("normal image"); axs[0].axis("off")
axs[1].imshow(_fish); axs[1].set_title("synthetic fisheye (lambda=0.5)"); axs[1].axis("off")
plt.tight_layout(); plt.show()


In [ ]:
class DistortionRegressionLoss(nn.Module):
    # Robust regression loss to the known synthetic division-model coefficient.
    def forward(self, pred, target):
        return F.smooth_l1_loss(pred.reshape(-1), target.reshape(-1))


def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running = 0.0
    n = 0
    for i, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad()
        out = model(inputs)
        loss = criterion(out, targets)
        loss.backward()
        optimizer.step()
        running += loss.item()
        n += 1
        if (i+1) % 50 == 0:
            print(f"  iter {i+1}/{len(loader)}  loss {running/n:.6f}")
    return running / max(n, 1)


@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    running = 0.0
    n = 0
    for inputs, targets in loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        out = model(inputs)
        loss = criterion(out, targets)
        running += loss.item()
        n += 1
    return running / max(n, 1)


## 5. Train (resume automatically from `checkpoint.pth.tar`)
The checkpoint stores epoch, losses, model weights, optimizer state and config. Rerunning this
cell after stopping continues from the last saved epoch. Set `CFG["EPOCHS_TO_RUN"] = None` to
train until `MAX_EPOCH`; otherwise it trains only that many epochs per notebook execution.


In [ ]:
# Make sure the dataset cell has run and produced image lists.
# Auto-load the NYU images if the discovery cell has not run yet.
if "train_images" not in globals() or not train_images:
    print("train_images not defined - auto-loading NYU images now.")
    train_images, val_images, DATA_ROOT = load_nyu_images(
        max_train=CFG.get("MAX_TRAIN_IMAGES"),
        max_val=CFG.get("MAX_VAL_IMAGES"),
        val_ratio=CFG.get("VAL_RATIO", 0.1),
    )

if "model" not in globals() or model is None:
    raise RuntimeError("model is not defined. Run the Model cell before training.")

transform = model.get_transforms()

distortions = get_distortions(CFG["NUM_DISTORTIONS"], random_values=False)

train_set = DistortDataset(train_images, CFG["IMAGE_HEIGHT"], CFG["IMAGE_WIDTH"], transform,
                           distortions, return_distortion=True)
val_set   = DistortDataset(val_images,   CFG["IMAGE_HEIGHT"], CFG["IMAGE_WIDTH"], transform,
                           distortions, return_distortion=True)

train_loader = DataLoader(train_set, batch_size=CFG["BATCH_SIZE"], shuffle=True,
                          num_workers=CFG["NUM_WORKERS"], drop_last=True)
val_loader   = DataLoader(val_set,   batch_size=CFG["BATCH_SIZE"], shuffle=False,
                          num_workers=CFG["NUM_WORKERS"], drop_last=False)

optimizer = optim.Adam(model.parameters(), lr=CFG["LEARNING_RATE"])
criterion = DistortionRegressionLoss().to(DEVICE)

start_epoch = 0
train_losses, val_losses = [], []
if os.path.exists(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt["state_dict"])
    optimizer.load_state_dict(ckpt["optimizer"])
    start_epoch = ckpt["epoch"]           # number of already-completed epochs
    train_losses = ckpt.get("train_losses", [])
    val_losses   = ckpt.get("val_losses", [])
    print("Resumed checkpoint:", CHECKPOINT_PATH)
    print("  already trained epochs:", start_epoch)

end_epoch = CFG["MAX_EPOCH"]
if CFG["EPOCHS_TO_RUN"] is not None:
    end_epoch = min(end_epoch, start_epoch + int(CFG["EPOCHS_TO_RUN"]))

if end_epoch <= start_epoch:
    print("Already trained through epoch", end_epoch, "(nothing to do). Increase MAX_EPOCH if needed.")
else:
    for epoch in range(start_epoch, end_epoch):
        print(f"Epoch {epoch+1}/{CFG['MAX_EPOCH']}")
        tl = train_one_epoch(model, train_loader, criterion, optimizer)
        vl = validate(model, val_loader, criterion)
        train_losses.append(tl)
        val_losses.append(vl)
        print(f"  train {tl:.6f}  val {vl:.6f}  (best-val so far {min(val_losses):.6f})")

        # plot train/val loss
        plt.figure(figsize=(9, 5))
        plt.plot(range(1, len(train_losses)+1), train_losses, "o-", label="train")
        plt.plot(range(1, len(val_losses)+1),   val_losses,   "s-", label="val")
        plt.yscale("log")
        plt.xlabel("epoch"); plt.ylabel("loss")
        plt.title("Fisheye distortion estimation loss")
        plt.legend(); plt.grid(True, alpha=0.4)
        plt.tight_layout()
        plt.savefig(LOSS_PLOT_PATH)
        plt.close()
        ipd.display(Image.open(LOSS_PLOT_PATH))

        # curriculum schedule from the original repo
        if CFG["CURRICULUM_ENABLED"] and (epoch+1) % CFG["SWITCH_EPOCH"] == 0:
            num_patterns = min(10, int((epoch+1)/CFG["SWITCH_EPOCH"]) + 2)
            if num_patterns < 10:
                d2 = get_distortions(num_patterns, random_values=False)
            else:
                d2 = get_distortions(10, random_values=True)
            train_set.set_distortions(d2)
            val_set.set_distortions(d2)
            print("  curriculum: switched to", len(d2), "distortion values")

        # save checkpoint so we can stop and resume
        torch.save({
            "epoch": epoch+1,
            "train_losses": train_losses,
            "val_losses": val_losses,
            "state_dict": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "cfg": CFG,
        }, CHECKPOINT_PATH)
        print("  saved checkpoint ->", CHECKPOINT_PATH)


## 6. Plot train / val loss
If you returned to the notebook later (after training more epochs), this re-reads the checkpoint
and plots the full history.


In [ ]:
if os.path.exists(CHECKPOINT_PATH):
    ck = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    tl = ck.get("train_losses", [])
    vl = ck.get("val_losses", [])
    print("Trained epochs:", len(tl), "| last train loss:", (tl[-1] if tl else None), "| last val loss:", (vl[-1] if vl else None))
    plt.figure(figsize=(9, 5))
    if tl: plt.plot(range(1, len(tl)+1), tl, "o-", label="train")
    if vl: plt.plot(range(1, len(vl)+1), vl, "s-", label="val")
    if tl or vl:
        plt.yscale("log"); plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend()
        plt.grid(True, alpha=0.4); plt.tight_layout(); plt.savefig(LOSS_PLOT_PATH); plt.show()
else:
    print("No checkpoint found at", CHECKPOINT_PATH)


## 7. Kannala–Brandt parameter extraction
The model estimates one number: the division-model coefficient `λ` (the original network does
not directly output a full KB parameter set). To get the KB parameters
`(fx, fy, cx, cy, k1, k2, k3, k4)`, we sample the division-model radial mapping and
least-squares fit the KB polynomial

```
theta_d = theta * (1 + k1*theta^2 + k2*theta^4 + k3*theta^6 + k4*theta^8)
rho_d   = f * theta_d
```

This notebook uses the mean predicted `λ` over a validation sample. You can also hard-code
`LAMBDA_MANUAL` if you already know the coefficient.


In [ ]:
def division_to_kannala(distortion, image_w, image_h, style="min_axis"):
    # Fit Kannala-Brandt polynomial to the single-parameter division model.
    # Returns fx, fy, cx, cy and k1..k4, plus the division-model lambda used.
    # style: min_axis -> f = min(W,H)/2 (default), x_axis -> f = W/2,
    #        y_axis -> f = H/2, geo_mean -> f = sqrt(W*H)/2
    W, H = int(image_w), int(image_h)
    cx, cy = W/2.0, H/2.0
    if style == "min_axis":
        f = min(W, H)/2.0
    elif style == "x_axis":
        f = W/2.0
    elif style == "y_axis":
        f = H/2.0
    else:
        f = math.sqrt(W*H)/2.0

    d = _clipd(distortion)
    r_limit = min(1.0, 0.999/math.sqrt(max(d, 1e-6))) if d > 1e-9 else 1.0
    r = np.linspace(0.0, r_limit, 4001)
    denom = 1.0 - d*r*r
    valid = denom > 1e-8
    r = r[valid]
    if len(r) < 20:
        r = np.linspace(0.0, 1.0, 4001)
        denom = 1.0 - d*r*r
        r = r[denom > 1e-8]

    r_u = r / (1.0 - d*r*r)
    theta = np.arctan(r_u)
    keep = theta > 1e-8
    r, theta = r[keep], theta[keep]
    y = r/theta - 1.0
    X = np.stack([theta**2, theta**4, theta**6, theta**8], axis=1)
    k, *_ = np.linalg.lstsq(X, y, rcond=None)
    k1, k2, k3, k4 = [float(v) for v in k]

    # report the model error at the sample points (in normalized units)
    theta_d_fit = theta * (1.0 + k1*theta**2 + k2*theta**4 + k3*theta**6 + k4*theta**8)
    fit_error = float(np.mean(np.abs(theta_d_fit - r)))

    params = {
        "division_lambda": float(d),
        "image_width": W,
        "image_height": H,
        "fx": float(f), "fy": float(f),
        "cx": float(cx), "cy": float(cy),
        "k1": k1, "k2": k2, "k3": k3, "k4": k4,
        "fit_rms": fit_error,
    }
    return params, (r, theta, theta_d_fit)


@torch.no_grad()
def predict_distortion_for_path(model, image_path, height, width, transform):
    img = Image.open(image_path).convert("RGB").resize((width, height), Image.LANCZOS)
    x = transform(img).unsqueeze(0).to(DEVICE)
    return float(model(x).cpu().item())


# Auto-load the NYU images if the discovery/training cells have not run yet.
if "val_images" not in globals() or not val_images:
    print("val_images not defined - auto-loading NYU images now.")
    train_images, val_images, DATA_ROOT = load_nyu_images(
        max_train=CFG.get("MAX_TRAIN_IMAGES"),
        max_val=CFG.get("MAX_VAL_IMAGES"),
        val_ratio=CFG.get("VAL_RATIO", 0.1),
    )

# Prediction on a validation sample (used for KB extraction).
if "model" not in globals() or model is None:
    raise RuntimeError("model is not defined. Run the Model cell before KB extraction.")


model.eval()
if val_images:
    sample_preds = []
    for p in val_images[:50]:
        sample_preds.append(predict_distortion_for_path(model, p, CFG["IMAGE_HEIGHT"], CFG["IMAGE_WIDTH"], transform))
    mean_pred = float(np.mean(sample_preds))
    print("Mean predicted lambda over", len(sample_preds), "validation images:", round(mean_pred, 5))
    print("Range:", round(min(sample_preds),5), "..", round(max(sample_preds),5))
else:
    mean_pred = 0.5
    print("No validation images (using a placeholder lambda=0.5). Set LAMBDA_MANUAL below if needed.")

# You may override the value used for KB extraction here.
LAMBDA_MANUAL = None   # e.g. 0.5
if LAMBDA_MANUAL is not None:
    mean_pred = float(LAMBDA_MANUAL)

kb, (r, theta, theta_d_fit) = division_to_kannala(mean_pred, CFG["IMAGE_WIDTH"], CFG["IMAGE_HEIGHT"])
print("Division-model lambda:", round(kb["division_lambda"], 5))
for key in ["fx", "fy", "cx", "cy", "k1", "k2", "k3", "k4"]:
    print(f"  {key}: {kb[key]:.8f}")
print("KB fit RMS (normalized):", round(kb["fit_rms"], 6))

with open(KB_JSON_PATH, "w") as f:
    json.dump(kb, f, indent=2, default=str)
print("Saved KB camera model ->", KB_JSON_PATH)

plt.figure(figsize=(8, 5))
plt.plot(theta, r, label="division model", lw=2)
plt.plot(theta, theta_d_fit, "--", label="KB fit", lw=2)
plt.xlabel("theta (rad)"); plt.ylabel("theta_d / distorted radius")
plt.title("Division-model vs Kannala-Brandt radial mapping")
plt.legend(); plt.grid(True, alpha=0.4); plt.tight_layout(); plt.show()


## 8. Inference on your own images
Put your photos in a folder (in a Kaggle notebook, upload them as a dataset or add an input),
then set `INFERENCE_INPUT_DIR` to that folder. The notebook predicts `λ` for every image and
writes the rectified result to `inference_out/`.


In [ ]:
INFERENCE_INPUT_DIR = "/kaggle/input/my_images"   # <-- CHANGEME: folder with YOUR images
INFERENCE_OUTPUT_DIR = os.path.join(WORK_BASE, "inference_out")
os.makedirs(INFERENCE_OUTPUT_DIR, exist_ok=True)

# Re-build model and load checkpoint for inference.
inf_model = ParametersEstimationModule().to(DEVICE)
inf_model.eval()
if os.path.exists(CHECKPOINT_PATH):
    ck = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    inf_model.load_state_dict(ck["state_dict"])
    print("Loaded checkpoint:", CHECKPOINT_PATH, "| epochs:", ck.get("epoch"))
else:
    print("WARNING: no checkpoint at", CHECKPOINT_PATH, "- using random weights.")


def build_simple_rectify_map(dist, in_w, in_h, out_w=None, out_h=None):
    # Build a remap that rectifies a division-model fisheye image.
    # For every output (rectified) pixel we compute the distorted position where that
    # ray lands in the input image and sample it.
    out_w = out_w or in_w
    out_h = out_h or in_h
    f_in  = min(in_w, in_h) / 2.0
    f_out = min(out_w, out_h) / 2.0
    cx_in, cy_in = in_w/2.0, in_h/2.0
    cx_out, cy_out = out_w/2.0, out_h/2.0

    xs = (np.arange(out_w, dtype=np.float32)[None, :] - cx_out) / f_out
    ys = (np.arange(out_h, dtype=np.float32)[:, None] - cy_out) / f_out
    r = np.sqrt(xs*xs + ys*ys)
    xd, yd = division_forward_vec(xs, ys, r, dist)   # original -> distorted normalized
    map_x = xd*f_in + cx_in
    map_y = yd*f_in + cy_in
    return map_x.astype(np.float32), map_y.astype(np.float32)


def rectify_division(image_path, dist, out_w=None, out_h=None):
    img = cv2.imread(str(image_path))
    if img is None:
        return None, None
    h, w = img.shape[:2]
    map_x, map_y = build_simple_rectify_map(dist, w, h, out_w or w, out_h or h)
    rect = cv2.remap(img, map_x, map_y, cv2.INTER_LINEAR,
                     borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    return rect, dist


def rectify_kb(image_path, kb_params, out_w=None, out_h=None):
    # Optional rectification using OpenCV's fisheye module and the fitted KB params.
    K = np.array([[kb_params["fx"], 0, kb_params["cx"]],
                  [0, kb_params["fy"], kb_params["cy"]],
                  [0, 0, 1]], dtype=np.float64)
    D = np.array([kb_params["k1"], kb_params["k2"], kb_params["k3"], kb_params["k4"]], dtype=np.float64)
    img = cv2.imread(str(image_path))
    if img is None:
        return None
    h, w = img.shape[:2]
    out_w = out_w or w
    out_h = out_h or h
    map_x, map_y = cv2.fisheye.initUndistortRectifyMap(K, D, np.eye(3), K, (out_w, out_h), cv2.CV_32FC1)
    return cv2.remap(img, map_x, map_y, cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)


def add_black_border(img, border_px=24):
    # Surround the image with a solid black frame so the rectified result has a
    # clear black border when displayed or saved.
    img = np.asarray(img)
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    h, w = img.shape[:2]
    framed = np.zeros((h + 2*border_px, w + 2*border_px, 3), dtype=img.dtype)
    framed[border_px:border_px+h, border_px:border_px+w] = img
    return framed


# Collect your images.
user_exts = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")
user_images = []
if os.path.isdir(INFERENCE_INPUT_DIR):
    for dp, _, fns in os.walk(INFERENCE_INPUT_DIR):
        for fn in fns:
            if fn.lower().endswith(user_exts):
                user_images.append(os.path.join(dp, fn))
elif os.path.isfile(INFERENCE_INPUT_DIR):
    user_images = [INFERENCE_INPUT_DIR]

if not user_images:
    print("No images found in", INFERENCE_INPUT_DIR)
    print("Upload your images (e.g. add a Kaggle input dataset), then edit INFERENCE_INPUT_DIR and rerun.")
else:
    print("Found", len(user_images), "inference image(s) in", INFERENCE_INPUT_DIR)
    for ip in user_images:
        pil = Image.open(ip).convert("RGB").resize((CFG["IMAGE_WIDTH"], CFG["IMAGE_HEIGHT"]), Image.LANCZOS)
        xt = transform(pil).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            dist_pred = float(inf_model(xt).cpu().item())
        rect, _ = rectify_division(ip, dist_pred)
        out_name = os.path.join(INFERENCE_OUTPUT_DIR, os.path.splitext(os.path.basename(ip))[0] + "_rect.png")
        if rect is not None:
            rect_framed = add_black_border(rect)
            cv2.imwrite(out_name, rect_framed)
            print(f"  {os.path.basename(ip)}  predicted lambda={dist_pred:.5f}  ->  {out_name}")

    # show a few examples side by side
    n_show = min(3, len(user_images))
    fig, axs = plt.subplots(n_show, 2, figsize=(10, 3*n_show))
    # Flatten axes so indexing works both when n_show == 1 (1D array) and n_show > 1.
    axs_flat = np.atleast_2d(axs).reshape(-1)
    for ax in axs_flat:
        ax.axis("off")
    for i, ip in enumerate(user_images[:n_show]):
        pil = Image.open(ip).convert("RGB").resize((CFG["IMAGE_WIDTH"], CFG["IMAGE_HEIGHT"]), Image.LANCZOS)
        xt = transform(pil).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            dp_ = float(inf_model(xt).cpu().item())
        rect, _ = rectify_division(ip, dp_)
        axs_flat[i*2].imshow(cv2.cvtColor(cv2.imread(ip), cv2.COLOR_BGR2RGB))
        axs_flat[i*2].set_title(f"original (lambda={dp_:.3f})")
        if rect is not None:
            # Frame the rectified output with a black border for clearer presentation.
            rect_framed = add_black_border(rect)
            axs_flat[i*2+1].imshow(cv2.cvtColor(rect_framed, cv2.COLOR_BGR2RGB))
            axs_flat[i*2+1].set_title("rectified")
    plt.tight_layout(); plt.show()


### Where the outputs are
- `outputs/nyu/checkpoint.pth.tar` — model state + optimizer + loss history (resume point)
- `outputs/nyu/losses.png` — train/val loss plot
- `outputs/nyu/camera_kb.json` — Kannala–Brandt parameters
- `inference_out/*_rect.png` — rectified versions of your images

Tip: to resume training after stopping the Kaggle session, run cells again with
`CFG["EPOCHS_TO_RUN"]` set to the number of additional epochs you want. If `checkpoint.pth.tar`
was written to `/kaggle/working`, enable *Save outputs* so it survives a restart, or re-upload it.
